In [ ]:
from collections import defaultdict
from datetime import datetime
import json
with open("odds_matches.json") as f:
    matches = json.load(f)
bookies = defaultdict(list)
BOOKIES = {
    "163": "eFortuna",
    "165": "STS",
    "502": "LV BET",
    "572": "BETFAN",
    "591": "Superbet",
}
all_bookies = set()
def probablity_home_win(home_odds, away_odds):
    ph_raw = 1.0 / float(home_odds)
    pa_raw = 1.0 / float(away_odds)

    total = ph_raw + pa_raw

    return ph_raw/total
for match in matches:
    home_win = int(match["home-winner"] == "win")
    name = match["name"]
    home_score = int(match["homeResult"])
    away_score = int(match["awayResult"])
    home_odds, away_odds = match["odds"]
    match_date = datetime.datetime.fromtimestamp(match["date-start-base"]).date()
    bookies["all"].append({
        "date": ,
        "home_win": home_win,
        "home_odds": float(home_odds),
        "away_odds": float(away_odds),
        "BoN": max(home_score, away_score) * 2 - 1,
        "home_prob": probablity_home_win(home_odds, away_odds),
        "name": name
    })
    for provider, odds in match["openingsOdds"].items():
        home, away = odds
        if not (float(home) and float(away)):
            continue
        
        all_bookies.add(provider)
        if provider in BOOKIES:
            bookies[provider].append({
                "bookie": BOOKIES[provider],
                "home_win": home_win,
                "home_odds": float(home),
                "away_odds": float(away),
                "BoN": max(home_score, away_score) * 2 - 1,
                "home_prob": probablity_home_win(home, away),
                "name": name
            })

print(len(all_bookies))

5


In [2]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    log_loss,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score
)
import pandas as pd

for provider, records in bookies.items():
    df = pd.DataFrame(records)
    X = df["home_prob"]
    y = df["home_win"]
    size = len(X)
    auc = roc_auc_score(y, X)
    name = BOOKIES[provider]

    print(f"{name}(size={size}): AUC={auc:.3f}")
    for N in [1, 3, 5]:
        bon = df.loc[df["BoN"] == N]
        size = len(bon)
        X = bon["home_prob"]
        y = bon["home_win"]
        auc = roc_auc_score(y, X)
        print(f"Bo{N} (size={size}) AUC={auc:.3f}")

STS(size=865): AUC=0.792
Bo1 (size=192) AUC=0.777
Bo3 (size=520) AUC=0.808
Bo5 (size=153) AUC=0.761
LV BET(size=833): AUC=0.788
Bo1 (size=187) AUC=0.780
Bo3 (size=496) AUC=0.803
Bo5 (size=150) AUC=0.752
eFortuna(size=589): AUC=0.782
Bo1 (size=168) AUC=0.777
Bo3 (size=330) AUC=0.798
Bo5 (size=91) AUC=0.748
BETFAN(size=861): AUC=0.794
Bo1 (size=188) AUC=0.777
Bo3 (size=517) AUC=0.811
Bo5 (size=156) AUC=0.762
Superbet(size=849): AUC=0.792
Bo1 (size=179) AUC=0.779
Bo3 (size=514) AUC=0.809
Bo5 (size=156) AUC=0.757
